In [0]:
%pip install -U -qqqq databricks-sdk
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text(
    "lakebase-instance-name", "memory-lakebase", "lakebase-instance-name"
)
dbutils.widgets.text(
    name="DATABRICKS_CLIENT_ID", defaultValue="", label="DATABRICKS_CLIENT_ID"
)
dbutils.widgets.text(
    name="DATABRICKS_CLIENT_SECRET", defaultValue="", label="DATABRICKS_CLIENT_SECRET"
)
dbutils.widgets.text(name="secret_scope", defaultValue="", label="secret_scope")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

In [0]:
secret_scope_name = dbutils.widgets.get("secret_scope")

# if needed create a secret scope
try:
    w.secrets.create_scope(scope=secret_scope_name)
except:
    print(f"Using existing secret scope: {secret_scope_name}")

In [0]:
if dbutils.widgets.get("DATABRICKS_CLIENT_ID") == "":
    print("no DATABRICKS_CLIENT_ID is provided")
else:
    w.secrets.put_secret(
        scope=secret_scope_name,
        key="DATABRICKS_CLIENT_ID",
        string_value=dbutils.widgets.get("DATABRICKS_CLIENT_ID"),
    )
if dbutils.widgets.get("DATABRICKS_CLIENT_SECRET") == "":
    print("no DATABRICKS_CLIENT_SECRET is provided")
else:
    w.secrets.put_secret(
        scope=secret_scope_name,
        key="DATABRICKS_CLIENT_SECRET",
        string_value=dbutils.widgets.get("DATABRICKS_CLIENT_SECRET"),
    )
DATABRICKS_HOST = w.config.host
w.secrets.put_secret(
    scope=secret_scope_name, key="DATABRICKS_HOST", string_value=DATABRICKS_HOST
)

In [0]:
# from databricks.sdk.service.postgres import Project

# w.postgres.create_project()

In [0]:
from databricks.sdk.service.database import DatabaseInstance
from datetime import timedelta

database_instance_name = dbutils.widgets.get("lakebase-instance-name")
database_instance = DatabaseInstance(
    name=database_instance_name,
    capacity="CU_1",
    enable_pg_native_login=False,
)
w.database.create_database_instance_and_wait(
    database_instance=database_instance, timeout=timedelta(minutes=20)
)

In [0]:
from databricks.sdk.service.database import (
    DatabaseInstanceRole,
    DatabaseInstanceRoleAttributes,
    DatabaseInstanceRoleIdentityType,
    DatabaseInstanceRoleMembershipRole,
)

database_instance_role = DatabaseInstanceRole(
    attributes=DatabaseInstanceRoleAttributes(
        bypassrls=False,
        createdb=False,
        createrole=False,
    ),
    identity_type=DatabaseInstanceRoleIdentityType.SERVICE_PRINCIPAL,
    membership_role=DatabaseInstanceRoleMembershipRole.DATABRICKS_SUPERUSER,
    name=dbutils.widgets.get("DATABRICKS_CLIENT_ID"),
)
w.database.create_database_instance_role(
    instance_name=dbutils.widgets.get("lakebase-instance-name"),
    database_instance_role=database_instance_role,
)

In [0]:
from databricks.sdk.service.database import DatabaseCatalog

w.database.create_database_catalog(
    catalog=DatabaseCatalog(
        name=f"{database_instance_name}-catalog",
        database_instance_name=database_instance_name,
        database_name="databricks_postgres",
        create_database_if_not_exists=True,
    )
)